# Training Faster R-CNN with k-fold cross validation

The code in here is roughly based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Hyperparameters


In [20]:
BATCH_SIZE = 4

# Wether to use the COCO pretrained Faster R-CNN
PRETRAINED = True

# Wether to use the AGAR dataset for pretraining
AGAR_TRAINING = True
EPOCHS_AGAR = 1

EPOCHS_OWN = 10
KFOLDS_OWN = 5

# Max norm for gradient clipping
MAX_NORM = 1

In [21]:
from pathlib import Path
from typing import Optional

import torch
from engine import evaluate, train_one_epoch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_data: DataLoader,
        val_data: Optional[DataLoader],
        optimizer: torch.optim.Optimizer,
        gpu_id: int,
        writer: Optional[SummaryWriter],
        model_save_path: str,
    ) -> None:
        self.gpu_id = gpu_id
        self.model = model.to(gpu_id)
        self.train_data = train_data
        self.val_data = val_data
        self.optimizer = optimizer
        self.writer = writer

        self.should_val = val_data is not None

        self.model_save_path = model_save_path
        Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)

    def _run_epoch(self, epoch, scaler):
        # train for one epoch, printing every 10 iterations
        train_losses = train_one_epoch(
            self.model,
            self.optimizer,
            self.train_data,
            self.gpu_id,
            epoch,
            scaler=scaler,
            print_freq=10,
            max_norm=MAX_NORM,
        )

        avg_train_loss_classifier = sum(
            [loss["loss_classifier"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_box_reg = sum(
            [loss["loss_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_objectness = sum(
            [loss["loss_objectness"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_rpn_box_reg = sum(
            [loss["loss_rpn_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss = sum(
            [
                (
                    loss["loss_classifier"]
                    + loss["loss_box_reg"]
                    + loss["loss_objectness"]
                    + loss["loss_rpn_box_reg"]
                )
                for loss in train_losses
            ]
        ) / len(train_losses)

        # check loss on validation set
        if self.should_val:
            with torch.no_grad():
                val_losses = train_one_epoch(
                    self.model,
                    self.optimizer,
                    self.val_data,
                    self.gpu_id,
                    epoch,
                    print_freq=10,
                    optimize=False,
                )

                avg_val_loss_classifier = sum(
                    [loss["loss_classifier"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_box_reg = sum(
                    [loss["loss_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_objectness = sum(
                    [loss["loss_objectness"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_rpn_box_reg = sum(
                    [loss["loss_rpn_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss = sum(
                    [
                        (
                            loss["loss_classifier"]
                            + loss["loss_box_reg"]
                            + loss["loss_objectness"]
                            + loss["loss_rpn_box_reg"]
                        )
                        for loss in val_losses
                    ]
                ) / len(val_losses)

                # evaluate on the validation dataset
                coco_evaluator, mae, smape = evaluate(
                    self.model, self.val_data, device=self.gpu_id
                )

                stats = coco_evaluator.coco_eval["bbox"].stats

                avg_precision_iou_50 = stats[0]
                avg_recall_iou_50 = stats[8]

        if self.writer is not None:
            self.writer.add_scalar(
                f"avg_train_loss_classifier", avg_train_loss_classifier, epoch
            )
            self.writer.add_scalar(
                f"avg_train_loss_box_reg", avg_train_loss_box_reg, epoch
            )
            self.writer.add_scalar(
                f"avg_train_loss_objectness", avg_train_loss_objectness, epoch
            )
            self.writer.add_scalar(
                f"avg_train_loss_rpn_box_reg", avg_train_loss_rpn_box_reg, epoch
            )
            self.writer.add_scalar(f"avg_train_loss", avg_train_loss, epoch)

            self.writer.add_scalar(
                f"avg_val_loss_classifier", avg_val_loss_classifier, epoch
            )
            self.writer.add_scalar(f"avg_val_loss_box_reg", avg_val_loss_box_reg, epoch)
            self.writer.add_scalar(
                f"avg_val_loss_objectness", avg_val_loss_objectness, epoch
            )
            self.writer.add_scalar(
                f"avg_val_loss_rpn_box_reg", avg_val_loss_rpn_box_reg, epoch
            )
            self.writer.add_scalar(f"avg_val_loss", avg_val_loss, epoch)

            self.writer.add_scalar(f"avg_precision_iou_50", avg_precision_iou_50, epoch)
            self.writer.add_scalar(f"avg_recall_iou_50", avg_recall_iou_50, epoch)
            self.writer.add_scalar(f"mae", mae, epoch)
            self.writer.add_scalar(f"smape", smape, epoch)

            self.writer.flush()

    def _save_checkpoint(self, epoch):
        ckp = self.model.state_dict()
        torch.save(ckp, self.model_save_path)
        print(f"Epoch {epoch} | Training checkpoint saved at {self.model_save_path}")

    def train(self, max_epochs: int):
        scaler = torch.cuda.amp.GradScaler()

        for epoch in range(max_epochs):
            self._run_epoch(epoch, scaler)

            if epoch % 10 == 0:
                self._save_checkpoint(epoch)

        self._save_checkpoint(max_epochs)

## Training on the AGAR dataset

Simple training and validation run on the high-resolution images of the AGAR
dataset.


In [22]:
from datetime import datetime
import json

import utils
from dataset_agar import AgarBBDataset
from dataset_photobox import PhotoboxBBDataset
from model import get_model_instance_segmentation
from sklearn.model_selection import KFold
from torch.optim import AdamW
from torch.utils.data import Dataset


def prepare_agar_datasets():
    # The box data for these IDs of the AGAR dataset have negative x values for the bounding boxes.
    faulty_ids = ["5794", "11988"]

    with open("../AGAR_dataset/training_lists/higher_resolution_train.txt") as f:
        train_list = json.load(f)
        train_list = [
            int(id) for id in train_list if id.isdigit() and id not in faulty_ids
        ]

    with open("../AGAR_dataset/training_lists/higher_resolution_val.txt") as f:
        val_list = json.load(f)
        val_list = [int(id) for id in val_list if id.isdigit() and id not in faulty_ids]

    print(f"Training on {len(train_list)} images.")
    print(f"Validating on {len(val_list)} images.")

    train_dataset = AgarBBDataset("../AGAR_dataset/dataset", train_list, is_train=True)
    val_dataset = AgarBBDataset("../AGAR_dataset/dataset", val_list, is_train=False)

    return train_dataset, val_dataset


def prepare_dataloader(dataset: Dataset, batch_size: int, is_train: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=4,
        pin_memory=True,
        shuffle=is_train,
        collate_fn=utils.collate_fn,
    )


def main(device):
    start_time_str = datetime.now().strftime("%Y-%m-%d_%H-%M")

    # Agar Pretraining
    if AGAR_TRAINING:
        print("Training on AGAR dataset\n" + "-" * 10 + "\n")

        train_dataset, val_dataset = prepare_agar_datasets()

        train_loader = prepare_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_dataset = prepare_dataloader(val_dataset, BATCH_SIZE, is_train=False)

        model = get_model_instance_segmentation(
            AgarBBDataset.num_classes,
            pretrained=PRETRAINED,
        )

        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

        writer = SummaryWriter(f"runs/{start_time_str}/agar")
        agar_model_path = f"artifacts/{start_time_str}/agar.pth"

        trainer = Trainer(
            model,
            train_loader,
            val_dataset,
            optimizer,
            device,
            writer,
            agar_model_path,
        )

        trainer.train(EPOCHS_AGAR)

    # K-fold cross validation
    kfold = KFold(n_splits=KFOLDS_OWN, shuffle=True)

    full_train_dataset = PhotoboxBBDataset(
        "../train_data", is_train=True, patch_augmentation=False
    )

    full_val_dataset = PhotoboxBBDataset("../train_data", is_train=False)

    for fold, (train_ids, val_ids) in enumerate(kfold.split(full_train_dataset)):
        print(f"Training fold {fold}\n" + "-" * 10 + "\n")

        train_dataset = torch.utils.data.Subset(full_train_dataset, train_ids)
        val_dataset = torch.utils.data.Subset(full_val_dataset, val_ids)

        base_model = get_model_instance_segmentation(
            PhotoboxBBDataset.num_classes,
            pretrained=PRETRAINED if agar_model_path is None else False,
        ).to(device)
        if agar_model_path is not None:
            base_model.load_state_dict(torch.load(agar_model_path))

        train_loader = prepare_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_loader = prepare_dataloader(val_dataset, BATCH_SIZE, is_train=False)

        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

        writer = SummaryWriter(f"runs/{start_time_str}/fold_{fold}")

        trainer = Trainer(
            base_model,
            train_loader,
            val_loader,
            optimizer,
            device,
            writer,
            f"artifacts/{start_time_str}/fold_{fold}.pth",
        )

        trainer.train(EPOCHS_OWN)

    print(f"Final training run\n" + "-" * 10 + "\n")

    model = get_model_instance_segmentation(
        PhotoboxBBDataset.num_classes,
        pretrained=PRETRAINED if agar_model_path is None else False,
    ).to(device)
    if agar_model_path is not None:
        model.load_state_dict(torch.load(agar_model_path))

    train_loader = prepare_dataloader(full_train_dataset, BATCH_SIZE, is_train=True)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=0.005, weight_decay=0.01)

    writer = SummaryWriter(f"runs/{start_time_str}/final")

    trainer = Trainer(
        model,
        train_loader,
        None,
        optimizer,
        device,
        writer,
        f"artifacts/{start_time_str}/final.pth",
    )

    trainer.train(EPOCHS_OWN)

In [23]:
if __name__ == "__main__":
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    main(device)

Training on AGAR dataset
----------

Training on 5241 images.
Validating on 1746 images.


/home/ms/l/ld580269/miniconda3/lib/python3.12/site-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at /opt/conda/conda-bld/pytorch_1712608847532/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch: [0]  [   0/1311]  eta: 7:25:08  lr: 0.000010  loss: 3.9359 (3.9359)  loss_classifier: 0.6127 (0.6127)  loss_box_reg: 0.5020 (0.5020)  loss_objectness: 2.4147 (2.4147)  loss_rpn_box_reg: 0.4065 (0.4065)  time: 20.3723  data: 7.0779  max mem: 4226
Epoch: [0]  [  10/1311]  eta: 0:54:54  lr: 0.000060  loss: 1.5872 (1.7881)  loss_classifier: 0.3594 (0.3649)  loss_box_reg: 0.3371 (0.3685)  loss_objectness: 0.7597 (0.9088)  loss_rpn_box_reg: 0.0810 (0.1458)  time: 2.5320  data: 0.7334  max mem: 5665
Epoch: [0]  [  20/1311]  eta: 0:49:07  lr: 0.000110  loss: 0.8073 (1.3145)  loss_classifier: 0.1888 (0.2610)  loss_box_reg: 0.3297 (0.3511)  loss_objectness: 0.2814 (0.5764)  loss_rpn_box_reg: 0.0741 (0.1260)  time: 1.3786  data: 0.7461  max mem: 5665
Epoch: [0]  [  30/1311]  eta: 0:43:05  lr: 0.000160  loss: 0.5992 (1.0814)  loss_classifier: 0.0982 (0.2118)  loss_box_reg: 0.2932 (0.3276)  loss_objectness: 0.1063 (0.4240)  loss_rpn_box_reg: 0.0808 (0.1180)  time: 1.7357  data: 1.1256  max m

KeyboardInterrupt: 